# Outline

This notebook demonstrates how to build a pipeline to predict results using Optuna hyperparameter lookup.

Preprocess and feature Engineering is base on previous notebook.

EDA link: [EDA of S6E2](https://www.kaggle.com/code/wesleyhuan/basic-eda-season-6-episode-2)

**Key point: This is a classification problem, but the submission we returned looks like this. This means we're trying to predict the probability of heart disease. Therefore, you might need to use a different evaluation metric, such as mean squared error (MSE) instead of the area under the curve (AUC).**

Submission:

id,Heart Disease

630000,0.2

630001,0.3

630002,0.1

etc.

Steps:

1. Load Data
2. Preprocess Data
3. Train model with Hyper parameter (OPTUNA)
4. Submission

In [1]:
# import necessary library
import numpy as np
import pandas as pd 
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import OneHotEncoder,StandardScaler

from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split

#for optuna pipline
import optuna
from sklearn.pipeline import Pipeline
from sklearn.model_selection import KFold, cross_val_score
from sklearn.base import BaseEstimator, TransformerMixin
import torch

In [2]:
# config
class CFG:
    train_csv = '/kaggle/input/playground-series-s6e2/train.csv'
    test_csv = '/kaggle/input/playground-series-s6e2/test.csv'
    sample_submission_csv = '/kaggle/input/playground-series-s6e2/sample_submission.csv'
    N_FOLDS = 5
    RANDOM_SEED = 42
    
#torch.device('cuda' if torch.cuda.is_available() else 'cpu')
#device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Load Data

In [3]:
train = pd.read_csv(CFG.train_csv)
test = pd.read_csv(CFG.test_csv)

In [4]:
train.head()

,id,Age,Sex,Chest pain type,BP,Cholesterol,FBS over 120,EKG results,Max HR,Exercise angina,ST depression,Slope of ST,Number of vessels fluro,Thallium,Heart Disease
0,0,58,1,4,152,239,0,0,158,1,3.6,2,2,7,Presence
1,1,52,1,1,125,325,0,2,171,0,0.0,1,0,3,Absence
2,2,56,0,2,160,188,0,2,151,0,0.0,1,0,3,Absence
3,3,44,0,3,134,229,0,2,150,0,1.0,2,0,3,Absence
4,4,58,1,4,140,234,0,2,125,1,3.8,2,3,3,Presence


In [5]:
test.head()

,id,Age,Sex,Chest pain type,BP,Cholesterol,FBS over 120,EKG results,Max HR,Exercise angina,ST depression,Slope of ST,Number of vessels fluro,Thallium
0,630000,58,1,3,120,288,0,2,145,1,0.8,2,3,3
1,630001,55,0,2,120,209,0,0,172,0,0.0,1,0,3
2,630002,54,1,4,120,268,0,0,150,1,0.0,2,3,7
3,630003,44,0,3,112,177,0,0,168,0,0.9,1,0,3
4,630004,43,1,1,138,267,0,0,163,0,1.8,2,0,7


# Preprocess Data

In [6]:
class Preprocessor:
    def __init__(self):
        self.medians = {}
        self.encoders = {}
        self.numeric_cols = []
        self.all_non_numeric = []
        self.scaler = StandardScaler() # initial std scaler
        self.scaling_cols = ['Age', 'BP', 'Cholesterol', 'Max HR', 'ST depression', 'BP_Age_Product']
        self.categorical_cols = ['Chest pain type', 'Thallium', 'Slope of ST', 'EKG results']
        self.level_mapping = {'Absence': 0, 'Presence': 1}
        self.level_cols = ['Heart Disease']

        
    def create_interaction_features(self, df):# add because of the Correlation matrix
        df = df.copy()
        
        # Age * BP
        if 'Age' in df.columns and 'BP' in df.columns:
            df['BP_Age_Product'] = df['Age'] * df['BP']
        
        return df
        
    def fit(self, df, y=None):
        """
        Learn the parameters (medians, categories) from the TRAINING data.
        """
        ##############df_temp = self.create_interaction_features(df)
        # Identify columns
        self.numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
        
        # Learn Medians for numeric columns
        for col in self.numeric_cols:
            self.medians[col] = df[col].median()
            
        # One hot encode the faeture 
        self.one_hot_encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
        self.one_hot_encoder.fit(df[self.categorical_cols].astype(str))

        # StandardScaler
        existing_scaling_cols = [c for c in self.scaling_cols if c in df.columns]
        if existing_scaling_cols:
            self.scaler.fit(df[existing_scaling_cols].fillna(self.medians))
            
        return self

    def transform(self, df):
        """
        Apply the learned parameters to the data (Train or Test).
        """
        df = df.copy()

        #############df = self.create_interaction_features(df)
        # Drop irrelevant columns (ID is usually dropped, Target handled separately)
        # Note: We don't drop target here to keep X and y aligned until the end
        if 'id' in df.columns:
            df = df.drop(columns=['id'])

        # level encoding for heart disease
        for col in self.level_cols:
            if col in df.columns:
                if not np.issubdtype(df[col].dtype, np.number):
                    df[col] = df[col].map(self.level_mapping).fillna(0).astype(int)
                else:
                    df[col] = df[col].fillna(0).astype(int)
        # Impute Missing Values using LEARNED medians
        for col in self.numeric_cols:
            if col in df.columns:
                df[col] = df[col].fillna(self.medians.get(col, 0))

        # Get one hot encode feature
        encoded_array = self.one_hot_encoder.transform(df[self.categorical_cols].astype(str))
        encoded_cols = self.one_hot_encoder.get_feature_names_out(self.categorical_cols)
        encoded_df = pd.DataFrame(encoded_array, columns=encoded_cols, index=df.index)
        
        # concat into original df
        df = pd.concat([df, encoded_df], axis=1)
        df = df.drop(columns=self.categorical_cols)

        # StandardScaler
        # make sure Age, BP feature is Standardlize
        target_scaling = [c for c in self.scaling_cols if c in df.columns]
        if target_scaling:
            df[target_scaling] = self.scaler.transform(df[target_scaling])
            
        return df


In [7]:
preprocessor = Preprocessor()

train_raw = train.drop(columns=['id'])

# learn the rule (median value...)
preprocessor.fit(train_raw)

# preprocess data
X_train_processed = preprocessor.transform(train_raw)

In [8]:
X_train_processed.head()

,Age,Sex,BP,Cholesterol,FBS over 120,Max HR,Exercise angina,ST depression,Number of vessels fluro,Heart Disease,...,Chest pain type_4,Thallium_3,Thallium_6,Thallium_7,Slope of ST_1,Slope of ST_2,Slope of ST_3,EKG results_0,EKG results_1,EKG results_2
0,0.467921,1,1.435822,-0.178490,0,0.271190,1,3.040655,2,1,...,1.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0
1,-0.258797,1,-0.367088,2.374837,0,0.951359,0,-0.754928,0,0,...,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
2,0.225682,0,1.970017,-1.692672,0,-0.095054,0,-0.754928,0,0,...,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
3,-1.227755,0,0.233882,-0.475388,0,-0.147375,0,0.299400,0,0,...,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
4,0.467921,1,0.634529,-0.326939,0,-1.455391,1,3.251520,3,1,...,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0


In [9]:
X_train_raw = train_raw.drop(columns=['Heart Disease'])
y_train = X_train_processed['Heart Disease']

# Train model with Hyper parameter (OPTUNA)

In [10]:
def XGB_objective(trial):
    model_params = {
        "n_estimators": trial.suggest_int("n_estimators", 300, 2000),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
    }
    
    pipeline = Pipeline(
        steps=[
            ("preprocess", Preprocessor()),
            ("model", XGBRegressor(
                objective="reg:logistic",
                random_state=CFG.RANDOM_SEED,
                n_jobs=-1,
                **model_params
            ))
        ]
    )

    scores = cross_val_score(
        pipeline,
        X_train_raw,
        y_train,
        cv=CFG.N_FOLDS,
        scoring="neg_root_mean_squared_error", # 改用 RMSE (負值是為了讓 Optuna 最大化)
        n_jobs=-1 
    )

    return -scores.mean()

In [11]:
study = optuna.create_study(direction="minimize")

# use Optuna to find best parameter
study.optimize(XGB_objective, n_trials=30)

[I 2026-02-10 15:53:52,568] A new study created in memory with name: no-name-736797e1-b266-465b-b53c-4cfdcda6e66a
[I 2026-02-10 15:55:39,290] Trial 0 finished with value: 0.2863613963127136 and parameters: {'n_estimators': 308, 'max_depth': 9, 'learning_rate': 0.016490809073873867, 'subsample': 0.5820466091273466, 'colsample_bytree': 0.6353249214632788, 'min_child_weight': 9, 'reg_alpha': 0.0006445229341647551, 'reg_lambda': 0.00026536012394400054}. Best is trial 0 with value: 0.2863613963127136.
[I 2026-02-10 16:02:05,826] Trial 1 finished with value: 0.28575687408447265 and parameters: {'n_estimators': 1600, 'max_depth': 9, 'learning_rate': 0.011229595127598802, 'subsample': 0.9353968263466055, 'colsample_bytree': 0.843707795294952, 'min_child_weight': 3, 'reg_alpha': 1.3982279159188722e-08, 'reg_lambda': 1.1667927911108464}. Best is trial 1 with value: 0.28575687408447265.
[I 2026-02-10 16:05:39,417] Trial 2 finished with value: 0.285772967338562 and parameters: {'n_estimators': 130

In [12]:
print(f"Best AUC score: {study.best_value:.4f}")
print("Best parameters:", study.best_params)

Best AUC score: 0.2847
Best parameters: {'n_estimators': 853, 'max_depth': 3, 'learning_rate': 0.09743038309137159, 'subsample': 0.6739100939114254, 'colsample_bytree': 0.5885863384724501, 'min_child_weight': 3, 'reg_alpha': 6.734046383517411e-08, 'reg_lambda': 7.620094550071678e-08}


# Submission

In [13]:
final_pipeline = Pipeline(
    steps=[
        ("preprocess", Preprocessor()),
        ("model", XGBRegressor(
            **study.best_params,
            objective="reg:logistic",
            random_state=CFG.RANDOM_SEED,
            n_jobs=-1
        ))
    ]
)

final_pipeline.fit(X_train_raw, y_train)
test_id = test["id"]
test = test.drop(columns=['id'])
y_pred = final_pipeline.predict(test)

In [14]:
submission = pd.read_csv(CFG.sample_submission_csv)

submission = pd.DataFrame({
    'id': test_id, 
    'Heart Disease': y_pred   
})

submission.to_csv('submission.csv', index=False)

In [15]:
submission.head()

,id,Heart Disease
0,630000,0.945953
1,630001,0.008409
2,630002,0.988815
3,630003,0.004417
4,630004,0.178509
